In [44]:
import pandas as pd
import numpy as np
import pathlib
import os
import time
import polars as pl
from datetime import datetime
import re
import platform
import glob
from pathlib import Path

In [45]:
# ── HELPER FUNCTIONS ─────────────────────────────────────────────────────────

def convert_to_datetime(struct_time):
    return datetime(*struct_time[:6])


def unify_dtypes(dfs):
    """Cast conflicting column dtypes to Utf8 so concat won't fail."""
    all_columns = set(col for df in dfs for col in df.columns)
    col_dtypes = {
        col: {df[col].dtype for df in dfs if col in df.columns}
        for col in all_columns
    }
    return [
        df.with_columns([
            pl.col(col).cast(pl.Utf8)
            for col, dtypes in col_dtypes.items()
            if col in df.columns and len(dtypes) > 1
        ])
        for df in dfs
    ]


def input_data(data_dir):
    """Read all xlsx/csv files in a directory into a single Polars DataFrame."""
    list_files = []
    for filename in pathlib.Path(data_dir).glob('**/*.*'):
        suffix = filename.suffix.lower()
        if suffix not in ('.xlsx', '.csv'):
            continue
        try:
            export_time_dt = convert_to_datetime(time.localtime(os.path.getmtime(filename)))
            if suffix == '.xlsx':
                df = pl.read_excel(filename, infer_schema_length=0)
            else:
                try:
                    df = pl.read_csv(filename, infer_schema_length=0, encoding='utf-8')
                except Exception:
                    df = pl.read_csv(filename, infer_schema_length=0,
                                     encoding='ISO-8859-1', ignore_errors=True)
            df = df.with_columns([
                pl.lit(filename.stem).alias('sheet_name'),
                pl.lit(export_time_dt).alias('Export time'),
            ])
            list_files.append(df)
        except Exception as e:
            print(f"Error reading {filename}: {e}")

    if not list_files:
        return pl.DataFrame()
    list_files = unify_dtypes(list_files)
    return pl.concat(list_files, how='vertical')


def convert_to_time(df, columns):
    """Parse AM/PM time strings into Python time objects (vectorised)."""
    for col in columns:
        df[col] = pd.to_datetime(df[col], format='%I:%M %p', errors='coerce').dt.time
    return df


def create_datetime_vec(df):
    """
    Fully vectorised replacement for the row-wise create_datetime().
    Returns two Series: Datetime_Start_Action, Datetime_End_Action.
    """
    base = pd.to_datetime(df['Date'])

    def _combine(base_date, time_series):
        """Combine a date Series with a time Series → Timestamp Series."""
        return pd.to_datetime(
            base_date.astype(str) + ' ' + time_series.astype(str),
            errors='coerce'
        )

    start = _combine(base, df['Start_Action'])
    end   = _combine(base, df['End_Action'])
    shift = _combine(base, df['Start_Shift'])

    # Add a day when the action wraps past midnight relative to shift start
    overnight_start = (df['Scheduled'] != 0) & start.notna() & shift.notna() & (start < shift)
    overnight_end   = (df['Scheduled'] != 0) & end.notna()   & shift.notna() & (end   < shift)
    start = start + pd.to_timedelta(overnight_start.astype(int), unit='D')
    end   = end   + pd.to_timedelta(overnight_end.astype(int),   unit='D')

    start = start.where(df['Scheduled'] != 0, pd.NaT)
    end   = end  .where(df['Scheduled'] != 0, pd.NaT)
    return start, end


def calculate_total_time(df, activity_list, time_column_name):
    """Sum Duration (hours) for given activities, grouped by Date+IEX_ID."""
    totals = (
        df[df['Scheduled Activity'].isin(activity_list)]
        .groupby(['Date', 'IEX_ID'], sort=False)['Duration']
        .sum()
        .div(3600)
        .reset_index(name=time_column_name)
    )
    return totals


def combine_ot_range(values):
    """Collapse a Series of OT ranges into a comma-separated unique string."""
    non_null = [v for v in values if v is not None and v != ""]
    distinct = list(dict.fromkeys(non_null))   # preserves order, removes dups
    return ",".join(distinct) if distinct else None


def calculate_ot_pre_post_shift(df):
    """
    Vectorised replacement for the row-wise calculate_ot_pre_post_shift().
    Groups extra-hours by Date+IEX_ID, joins original shift boundaries,
    and splits into pre- / post-shift OT hours.
    """
    extra = df[df['Scheduled Activity'] == 'Extra Hours'].copy()

    # Original shift boundaries per (Date, IEX_ID)
    shift_bounds = (
        df.groupby(['Date', 'IEX_ID'], sort=False)
        .agg(
            shift_start=('Datetime_Original_Start_Shift', 'min'),
            shift_end  =('Datetime_Original_End_Shift',   'max'),
        )
        .reset_index()
    )

    extra = extra.merge(shift_bounds, on=['Date', 'IEX_ID'], how='left')

    pre_mask  = extra['Datetime_Start_Action'] < extra['shift_start']
    post_mask = extra['Datetime_End_Action']   > extra['shift_end']

    extra['pre_sec']  = np.where(pre_mask,  extra['Duration'], 0)
    extra['post_sec'] = np.where(post_mask, extra['Duration'], 0)

    ot_agg = (
        extra.groupby(['Date', 'IEX_ID'], sort=False)
        .agg(OT_Pre=('pre_sec', 'sum'), OT_Post=('post_sec', 'sum'))
        .reset_index()
        .rename(columns={'OT_Pre': 'OT PreShift', 'OT_Post': 'OT PostShift'})
    )
    ot_agg['OT PreShift']  /= 3600
    ot_agg['OT PostShift'] /= 3600

    return df.merge(ot_agg, on=['Date', 'IEX_ID'], how='left')


def determine_ot_type(shift_tracking, ot_pre, ot_post):
    """Vectorised OT-type classification."""
    conditions = [
        shift_tracking == 'PO',
        (shift_tracking == 'PR - OT') & (ot_pre > 0) & (ot_post > 0),
        (shift_tracking == 'PR - OT') & (ot_pre > 0),
        (shift_tracking == 'PR - OT') & (ot_post > 0),
    ]
    choices = ['OT - PO', 'OT - Pre/Post Shift', 'OT - Pre Shift', 'OT - Post Shift']
    return np.select(conditions, choices, default='No OT')


def label_breaks(group):
    """Number Break activities within a (Date, IEX_ID) group."""
    mask = group['Scheduled Activity'] == 'Break'
    group.loc[mask, 'Scheduled Activity'] = (
        'Break_' + mask.cumsum().astype(str)
    )
    return group

In [46]:
# ── PATHS ────────────────────────────────────────────────────────────────────

first_glob = os.path.expanduser('~').replace('\\', '/')
test_path  = f'{first_glob}/Concentrix Corporation'
if not os.path.exists(test_path):
    raise FileNotFoundError(f'Not found the path: {test_path}')

folder_paths = {
    'input_iex'           : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_IEX_FOR_REPORT',
    'input_hc_master'     : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx',
    'output_iex'          : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_FOR_REPORT',
    'output_iex_all'      : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX',
    'output_iex_intervals': f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS',
    'output_iex_base'     : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_BASE',
    'hc_extend_by_month'  : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month',
    'input_iex_schedule_adjustment'     : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/IEX Backdated Schedule Adjustment Log.xlsx',
}
print('--- FULL FOLDER PATHS LIST ---')
for k, v in folder_paths.items():
    print(f'{k}: {v}')

--- FULL FOLDER PATHS LIST ---
input_iex: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_IEX_FOR_REPORT
input_hc_master: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx
output_iex: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_FOR_REPORT
output_iex_all: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_AGENT_IEX
output_iex_intervals: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS
output_iex_base: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_BASE
hc_extend_by_month: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month
input_iex_schedule_adjustment: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/IEX Back

In [47]:
# ── LOAD INPUT DATA ───────────────────────────────────────────────────────────

IEX_Input = input_data(folder_paths['input_iex']).to_pandas()

HC_MASTER_DATABASE = (
    input_data(folder_paths['hc_extend_by_month'])
    .to_pandas()
    .rename(columns={'IEX ID': 'IEX_ID'})
)

In [48]:
# ── CLEAN & RESHAPE IEX INPUT ─────────────────────────────────────────────────

DROP_COLS = ['__UNNAMED__4']
RENAME_MAP = {
    'Agent Schedules': 'Agent',
    '__UNNAMED__1'   : 'Date',
    '__UNNAMED__2'   : 'Start_Shift',
    '__UNNAMED__3'   : 'End_Shift',
    '__UNNAMED__5'   : 'Scheduled Activity',
    '__UNNAMED__6'   : 'Start_Action',
    '__UNNAMED__9'   : 'End_Action',
}

IEX = (
    IEX_Input
    .drop(columns=[c for c in DROP_COLS if c in IEX_Input.columns])
    .rename(columns=RENAME_MAP)
)

# Extract Generate Date from sentinel rows, then back-fill
IEX['Generate Date'] = np.where(
    IEX['Agent'].str.contains('Generation Date: ', na=False),
    IEX['Agent'].str.extract(r'Generation Date: (.+)')[0],
    np.nan,
)
IEX['Generate Date'] = IEX['Generate Date'].bfill()
IEX_edit = IEX.copy()
IEX_edit['Agent'] = IEX_edit['Agent'].ffill()
offTable_save = IEX_edit.copy()
offTable = offTable_save[offTable_save['Start_Shift'] == 'Off'].reset_index(drop=True)
IEX_edit = IEX_edit[
    (IEX_edit['Start_Shift'] != 'Off') &
    (IEX_edit['Date'] != 'Date') &
    ~(IEX_edit['Date'].isna() & IEX_edit['Scheduled Activity'].isna())
].reset_index(drop=True)
IEX_edit[['Date', 'Start_Shift', 'End_Shift']] = IEX_edit[['Date', 'Start_Shift', 'End_Shift']].ffill()
IEX_edit = IEX_edit[IEX_edit['Agent'].str.contains('Agent: ', na=False) | IEX_edit['Agent'].isna()].reset_index(drop=True)
IEX_edit = IEX_edit[IEX_edit['Scheduled Activity'].notna()].reset_index(drop=True)

offTable['Scheduled Activity'] = offTable['Scheduled Activity'].fillna(offTable['Start_Shift'])
offTable['Start_Shift']        = offTable['Start_Shift'].replace('Off', np.nan)

combined_df = pd.concat([offTable, IEX_edit], axis=0, ignore_index=True)

# ── LOAD SCHEDULE ADJUSTMENT LOG ──────────────────────────────────────────────

adj_log = pd.read_excel(
    folder_paths['input_iex_schedule_adjustment'],
    sheet_name='Log',
    header=0,
)

# ── NORMALIZE TIME COLUMNS TO MATCH combined_df FORMAT (h:MM AM/PM) ───────────
TIME_COLS = ['Start_Shift', 'End_Shift', 'Start_Action', 'End_Action']
_hour_fmt = '%#I:%M %p' if platform.system() == 'Windows' else '%-I:%M %p'
def to_ampm_str(val):
    if pd.isna(val) or val is None:
        return val
    if hasattr(val, 'hour') and hasattr(val, 'minute'):
        return f"{val.hour % 12 or 12}:{val.minute:02d} {'AM' if val.hour < 12 else 'PM'}"
    if isinstance(val, str):
        try:
            t = pd.to_datetime(val, format='%H:%M:%S')
            h = t.hour
            return f"{h % 12 or 12}:{t.minute:02d} {'AM' if h < 12 else 'PM'}"
        except Exception:
            return val
    return val

for col in TIME_COLS:
    if col in adj_log.columns:
        adj_log[col] = adj_log[col].apply(to_ampm_str)

# ── APPLY ADJUSTMENTS TO combined_df ──────────────────────────────────────────
combined_df['Date'] = pd.to_datetime(combined_df['Date'], errors='coerce')
adj_log['Date']     = pd.to_datetime(adj_log['Date'],     errors='coerce')
override_keys = set(zip(adj_log['Agent'], adj_log['Date']))
mask_to_drop = combined_df.apply(lambda row: (row['Agent'], row['Date']) in override_keys, axis=1)
combined_df_filtered = combined_df[~mask_to_drop].reset_index(drop=True)
adj_log = adj_log.reindex(columns=combined_df.columns)
combined_df = pd.concat([combined_df_filtered, adj_log],axis=0,ignore_index=True,)
combined_df = combined_df.sort_values(by=['Agent', 'Date', 'Start_Action'],na_position='first',).reset_index(drop=True)

def export_weekly_csvs(df: pd.DataFrame, output_dir: str = ".") -> None:
    os.makedirs(output_dir, exist_ok=True)
    df = df.copy()
    df['_date_parsed'] = pd.to_datetime(df['Date'], errors='coerce')
    df['_week_start'] = df['_date_parsed'].dt.to_period('W-SUN').apply(lambda p: p.start_time if pd.notna(p) else pd.NaT)
    exported = []
    for week_start, group in df.groupby('_week_start', dropna=False):
        out = group.drop(columns=['_date_parsed', '_week_start'])
        if pd.isna(week_start):
            filename = "IEX_Base_no_date.csv"
        else:
            filename = f"IEX_Base_{week_start.strftime('%Y_%m_%d')}.csv"
        filepath = os.path.join(output_dir, filename)
        out.to_csv(filepath, index=False, encoding='utf-8-sig')
        exported.append((filename, len(out)))
    print(f"✅ Exported {len(exported)} file(s) to '{output_dir}':")
    for fname, rows in exported:
        print(f"   {fname}  ({rows} rows)")

export_weekly_csvs(combined_df, output_dir=folder_paths['output_iex_base'])

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13744\111328250.py:41: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  offTable['Start_Shift']        = offTable['Start_Shift'].replace('Off', np.nan)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13744\111328250.py:75: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_df['Date'] = pd.to_datetime(combined_df['Date'], errors='coerce')


✅ Exported 20 file(s) to 'C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_BASE':
   IEX_Base_2026_01_05.csv  (4050 rows)
   IEX_Base_2026_01_12.csv  (4063 rows)
   IEX_Base_2026_01_19.csv  (3906 rows)
   IEX_Base_2026_01_26.csv  (3727 rows)
   IEX_Base_2026_02_02.csv  (3738 rows)
   IEX_Base_2026_02_09.csv  (3619 rows)
   IEX_Base_2026_02_16.csv  (3682 rows)
   IEX_Base_2026_02_23.csv  (3032 rows)
   IEX_Base_2026_03_02.csv  (3208 rows)
   IEX_Base_2026_03_09.csv  (2510 rows)
   IEX_Base_2026_03_16.csv  (3080 rows)
   IEX_Base_2026_03_23.csv  (3018 rows)
   IEX_Base_2026_03_30.csv  (2675 rows)
   IEX_Base_2026_04_06.csv  (2573 rows)
   IEX_Base_2026_04_13.csv  (3197 rows)
   IEX_Base_2026_04_20.csv  (3372 rows)
   IEX_Base_2026_04_27.csv  (3151 rows)
   IEX_Base_2026_05_04.csv  (3800 rows)
   IEX_Base_2026_05_11.csv  (3493 rows)
   IEX_Base_2026_05_18.csv  (3981 rows)


In [ ]:
# ── GET VALID WEEKS FROM input_iex ────────────────────────────────────────────
input_iex_files = (
    glob.glob(f'{folder_paths["input_iex"]}/*.csv') +
    glob.glob(f'{folder_paths["input_iex"]}/*.xlsx')
)

# Normalize stem: replace underscore with dash to parse as date
valid_weeks_dt = pd.to_datetime(
    [Path(f).stem.replace('_', '-') for f in input_iex_files],
    errors='coerce'
).dropna()
print(f"Valid weeks found: {sorted(valid_weeks_dt.strftime('%Y-%m-%d'))}")

# ── LOAD & PARSE combined_df ───────────────────────────────────────────────────
combined_df = input_data(folder_paths['output_iex_base']).to_pandas()

for col in ('Export time', 'Generate Date', 'Date'):
    combined_df[col] = pd.to_datetime(combined_df[col], errors='coerce')

# ── DERIVE Week_Monday (Monday of each Date) ───────────────────────────────────
combined_df['Week_Monday'] = (
    combined_df['Date']
    - pd.to_timedelta(combined_df['Date'].dt.dayofweek, unit='D')
).dt.normalize()

# ── FILTER: keep only rows whose week exists in input_iex ─────────────────────
combined_df = combined_df[combined_df['Week_Monday'].isin(valid_weeks_dt)].reset_index(drop=True)

print(f"Rows after filter: {len(combined_df)}")
print(f"Weeks retained: {sorted(combined_df['Week_Monday'].dt.strftime('%Y-%m-%d').unique())}")

for col in ('Export time', 'Generate Date', 'Date'):
    combined_df[col] = pd.to_datetime(combined_df[col], errors='coerce')

combined_df['Year']   = combined_df['Date'].dt.year
combined_df['Month']  = combined_df['Date'].dt.strftime('%b-%y')
combined_df['IEX_ID'] = combined_df['Agent'].str.extract(r'(\d+)', expand=False).astype(int)

# Keep only the latest Generate Date per (IEX_ID, Date)
max_gen = (
    combined_df.groupby(['IEX_ID', 'Date'], sort=False)['Generate Date']
    .max()
    .reset_index(name='Max Generate Date')
)
filtered = combined_df.merge(max_gen, on=['IEX_ID', 'Date'], how='left')
filtered = filtered[filtered['Generate Date'] == filtered['Max Generate Date']].copy()

# Scheduled flag
SCHEDULED_ACTIVITIES = {
    'Open Time', 'Extra Hours', 'No Call/No Show', 'PTO',
    'Training Offline', 'Sick Leave', 'Paid Leave', 'Termination',
    'Off Phone Misc', 'Billable Training', 'Nesting Training'
}
filtered['Scheduled'] = (
    filtered.groupby(['Date', 'IEX_ID'])['Scheduled Activity']
    .transform(lambda x: 1 if x.isin(SCHEDULED_ACTIVITIES).any() else 0)
)

Valid weeks found: ['2026-01-05', '2026-01-12', '2026-01-19', '2026-01-26', '2026-02-02', '2026-02-09', '2026-02-16', '2026-02-23', '2026-03-02', '2026-03-09', '2026-03-16', '2026-03-23', '2026-03-30', '2026-04-06', '2026-04-13', '2026-04-20', '2026-04-27', '2026-05-04', '2026-05-11', '2026-05-18']


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13744\2588571253.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined_df[col] = pd.to_datetime(combined_df[col], errors='coerce')


Rows after filter: 67875
Weeks retained: ['2026-01-05', '2026-01-12', '2026-01-19', '2026-01-26', '2026-02-02', '2026-02-09', '2026-02-16', '2026-02-23', '2026-03-02', '2026-03-09', '2026-03-16', '2026-03-23', '2026-03-30', '2026-04-06', '2026-04-13', '2026-04-20', '2026-04-27', '2026-05-04', '2026-05-11', '2026-05-18']


In [50]:
# ── TIME COLUMNS & DATETIME CONSTRUCTION ─────────────────────────────────────

time_cols = ['Start_Shift', 'End_Shift', 'Start_Action', 'End_Action']
filtered = convert_to_time(filtered, time_cols)

filtered['Datetime_Start_Action'], filtered['Datetime_End_Action'] = create_datetime_vec(filtered)

In [51]:
# ── SHIFT RANGE CALCULATIONS ──────────────────────────────────────────────────

def _shift_range(df, activity_filter, prefix):
    """Generic helper: min start / max end for a subset of activities."""
    subset = df[activity_filter]
    agg = (
        subset.groupby(['Date', 'IEX_ID'], sort=False)
        .agg(
            **{f'Datetime_{prefix}_Start_Shift': ('Datetime_Start_Action', 'min')},
            **{f'Datetime_{prefix}_End_Shift'  : ('Datetime_End_Action',   'max')},
        )
        .reset_index()
    )
    start_col = f'Datetime_{prefix}_Start_Shift'
    end_col   = f'Datetime_{prefix}_End_Shift'
    agg[f'{prefix} Shift'] = (
        agg[start_col].dt.strftime('%H%M') + '-' + agg[end_col].dt.strftime('%H%M')
    )
    return agg

fluctuate_mask = (
    filtered['Scheduled Activity'].isin(['Open Time', 'Extra Hours']) |
    filtered['Scheduled Activity'].str.contains('Training', na=False)
)
original_mask = (
    filtered['Scheduled Activity'].isin(['Open Time']) |
    filtered['Scheduled Activity'].str.contains('Training', na=False)
)

fluctuate_shifts = _shift_range(filtered, fluctuate_mask, 'Fluctuate')
first_shifts     = _shift_range(filtered, pd.Series(True, index=filtered.index), 'First')
original_shifts  = _shift_range(filtered, original_mask, 'Original')

result = (
    filtered
    .merge(fluctuate_shifts, on=['Date', 'IEX_ID'], how='left')
    .merge(first_shifts,     on=['Date', 'IEX_ID'], how='left')
    .merge(original_shifts,  on=['Date', 'IEX_ID'], how='left')
)

# Drop Lunch/Break rows only when there is no fluctuating shift
result = result[
    ~(
        result['Scheduled Activity'].isin(['Lunch', 'Break']) &
        result['Fluctuate Shift'].isna()
    )
].copy()

In [52]:
# ── DURATION & DERIVED COLUMNS ────────────────────────────────────────────────

result = result.sort_values('Export time').copy()

result['Night_Shift'] = (
    (result['Datetime_First_Start_Shift'].dt.hour >= 15) &
    (result['Datetime_First_End_Shift'].dt.date > result['Datetime_First_Start_Shift'].dt.date)
).astype(int)

result['Week_Monday'] = result['Date'] - pd.to_timedelta(result['Date'].dt.weekday, unit='d')
result['Week Begin']  = result['Week_Monday'].dt.strftime('WB%d%m')
result['Agent Name']  = result['Agent'].str.extract(r'\d+ (.+)', expand=False).str.upper()
result['Duration']    = (
    result['Datetime_End_Action'] - result['Datetime_Start_Action']
).dt.total_seconds()

result = result.drop_duplicates()

# ── ACTIVITY TIME TOTALS (vectorised, single pass) ────────────────────────────
TRAINING_ACTIVITIES = ['Training Offline', 'Training', 'Training Nesting', 'Nesting Training']
ACTIVITY_MAP = {
    'Open Time'      : 'Open Time',
    'Break'          : 'Break Time',
    'Lunch'          : 'Lunch Time',
    'Extra Hours'    : 'Extra Time',
    'No Call/No Show': 'NCNS',
    'PTO'            : 'AL',
}

# Merge training total first
result = result.merge(
    calculate_total_time(result, TRAINING_ACTIVITIES, 'Training'),
    on=['Date', 'IEX_ID'], how='left'
)

# Merge remaining activity totals in one pass
time_totals = (
    result[result['Scheduled Activity'].isin(ACTIVITY_MAP.keys())]
    .groupby(['Date', 'IEX_ID', 'Scheduled Activity'], sort=False)['Duration']
    .sum()
    .div(3600)
    .reset_index()
    .assign(**{'col_name': lambda d: d['Scheduled Activity'].map(ACTIVITY_MAP)})
    .pivot_table(index=['Date', 'IEX_ID'], columns='col_name', values='Duration', aggfunc='sum')
    .reset_index()
)
# Rename axis leftover
time_totals.columns.name = None
result = result.merge(time_totals, on=['Date', 'IEX_ID'], how='left')

result['Target'] = (
    result.get('Open Time', pd.Series(np.nan, index=result.index)).fillna(0) +
    result.get('Extra Time', pd.Series(np.nan, index=result.index)).fillna(0)
).replace(0, np.nan)

# Restore NaN for agents with neither Open Time nor Extra Time
result['Target'] = np.where(
    result.get('Open Time', np.nan).isna() & result.get('Extra Time', np.nan).isna(),
    np.nan,
    result['Target'],
)

result['Time_Of_Day'] = (
    result['Datetime_First_End_Shift'] - result['Datetime_First_Start_Shift']
).dt.total_seconds() / 3600

result = result.sort_values(['Date', 'IEX_ID', 'Datetime_Start_Action'], na_position='last')
result['First_Scheduled_Activity'] = (
    result.groupby(['Date', 'IEX_ID'])['Scheduled Activity'].transform('first')
)

In [53]:
# ── SHIFT TRACKING ────────────────────────────────────────────────────────────

fsa  = result['First_Scheduled_Activity']
ot   = result.get('Open Time',  pd.Series(np.nan, index=result.index))
et   = result.get('Extra Time', pd.Series(np.nan, index=result.index))
ncns = result.get('NCNS',       pd.Series(0,      index=result.index))

conditions = [
    ((ot > 0) & (ncns > 3)) | (ncns <= 5),
    ((ot > 0) | ot.isna())  & (ncns > 5),
    (ot > 0)  & (et > 0),
    (ot == 0) & (et > 0),
    (ot > 0),
    (fsa == 'Extra Hours')  & ot.isna(),
    fsa.isin(['Holiday', 'Bereavement', 'Off', 'Off Phone Misc', 'Termination', 'Unscheduled']),
    fsa == 'PTO',
    fsa.isin(['Sickness', 'Sick Leave']),
    fsa.isin(['Training Offline']),
    fsa.isin(['Paid Leave']),
    fsa.isin(['Leave']),
    fsa.isin(['Termination']),
    fsa.isin(['Offline']),
]
choices = [
    'HDL', 'NCNS', 'PR - OT', 'PO', 'PR', 'PO',
    fsa, 'AL', 'SL', 'Training Offline', 'CO', 'LWP', 'Termination', 'Offline',
]

result['Shift Tracking'] = np.select(conditions, choices, default=fsa)

result['Fluctuate Shift'] = np.where(
    result['Time_Of_Day'].isna(), result['Scheduled Activity'], result['Fluctuate Shift']
)
result['First Shift'] = np.where(
    result['Time_Of_Day'].isna(), result['Scheduled Activity'], result['First Shift']
)

result['Night_Shift'] = np.where(result['Shift Tracking'].str.contains('PR', na=False),result['Night_Shift'],0.0)

In [54]:
# ── CONSTANTS ────────────────────────────────────────────────────────────────
_HHMM_RE = re.compile(r'^\d{4}$')
_MERGE_GAP_MINS = 80

def _to_mins(t: str) -> int:
    return int(t[:2]) * 60 + int(t[2:])

def _to_hhmm(m: int) -> str:
    m = m % 1440
    return f"{m // 60:02d}{m % 60:02d}"

def combine_ot_range(series) -> str | None:
    raw = [
        str(r) for r in series
        if r is not None and str(r) not in ('', 'None', 'nan') and '-' in str(r)
    ]
    if not raw:
        return None

    numeric, opaque = [], []
    for r in raw:
        parts = r.split('-', 1)
        if len(parts) == 2 and _HHMM_RE.match(parts[0]) and _HHMM_RE.match(parts[1]):
            numeric.append(r)
        else:
            opaque.append(r)

    if opaque:
        seen = dict.fromkeys(opaque)
        if numeric:
            seen.update(dict.fromkeys(numeric))
        return ", ".join(seen)

    # ---------------------------------------------------------
    # [UPDATED LOGIC] Convert to minutes & Handle Cross-Day 
    # ---------------------------------------------------------
    intervals = []
    day_offset = 0
    last_s = -1

    for r in numeric:
        s_str, e_str = r.split('-', 1)
        s, e = _to_mins(s_str), _to_mins(e_str)

        if e < s:
            e += 1440
        if last_s != -1 and s < last_s - 720:
            day_offset += 1440

        last_s = s
        intervals.append([s + day_offset, e + day_offset])

    # ---------------------------------------------------------
    # NO SORTING HERE! We trust the true chronological order from the dataframe.
    # ---------------------------------------------------------
    if not intervals:
        return None

    merged = [intervals[0]]
    for s, e in intervals[1:]:
        prev = merged[-1]
        if s - prev[1] <= _MERGE_GAP_MINS:
            prev[1] = max(prev[1], e)
        else:
            merged.append([s, e])

    result = [f"{_to_hhmm(s)}-{_to_hhmm(e)}" for s, e in merged]
    return ", ".join(dict.fromkeys(result))


# ── MAIN TRANSFORM ───────────────────────────────────────────────────────────

extra_mask = result['Scheduled Activity'] == 'Extra Hours'

result['OT Range'] = np.where(
    result['Shift Tracking'] == 'PO',
    result['Fluctuate Shift'],
    np.where(
        extra_mask,
        result['Datetime_Start_Action'].dt.strftime('%H%M') + '-' +
        result['Datetime_End_Action'].dt.strftime('%H%M'),
        None,
    ),
)

result = result.sort_values(['Date', 'IEX_ID', 'Datetime_Start_Action'])

_ot_agg = (
    result[result['OT Range'].notna()]
    .groupby(['Date', 'IEX_ID'], sort=False)['OT Range']
    .agg(combine_ot_range)
    .reset_index()
    .rename(columns={'OT Range': 'Combined OT Range'})
)
result = result.merge(_ot_agg, on=['Date', 'IEX_ID'], how='left')

In [55]:
# ── OT PRE / POST SHIFT ───────────────────────────────────────────────────────

result = calculate_ot_pre_post_shift(result)

result['Start Date'] = pd.to_datetime(result['Datetime_Start_Action']).dt.normalize()
result['End Date']   = pd.to_datetime(result['Datetime_End_Action']).dt.normalize()

In [56]:
# ── COLUMN SELECTION FOR SORTED_DF ───────────────────────────────────────────

IEX_COLS = [
    'sheet_name', 'Generate Date', 'Year', 'Month', 'Week Begin', 'Week_Monday',
    'Date', 'Start Date', 'End Date', 'Agent Name', 'IEX_ID',
    'Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift', 'Fluctuate Shift',
    'Datetime_First_Start_Shift', 'Datetime_First_End_Shift', 'First Shift',
    'Scheduled Activity', 'Datetime_Start_Action', 'Datetime_End_Action',
    'Time_Of_Day', 'Duration', 'Open Time', 'Extra Time', 'Target',
    'Break Time', 'Lunch Time', 'Training', 'NCNS', 'AL', 'Night_Shift',
    'Scheduled', 'Shift Tracking', 'OT PreShift', 'OT PostShift', 'Combined OT Range',
]
sorted_df = result[[c for c in IEX_COLS if c in result.columns]].copy()

In [57]:
# ── MERGE WITH HC MASTER ──────────────────────────────────────────────────────

sorted_df['Date']   = pd.to_datetime(sorted_df['Date'])
sorted_df['IEX_ID'] = sorted_df['IEX_ID'].astype(str)

HC_MASTER_DATABASE['Date']   = pd.to_datetime(HC_MASTER_DATABASE['Date'])
HC_MASTER_DATABASE['IEX_ID'] = HC_MASTER_DATABASE['IEX_ID'].astype(str)

HC_COLS = ['Date', 'OracleID', 'People ID', 'IEX_ID', 'Employee Name', 'Email Id',
           'Alias', 'LOB', 'LOB_2', 'LOB_3', 'Supervisor Name', 'Wave',
           'Detail Status', 'Status']

sorted_df = (
    sorted_df
    .merge(HC_MASTER_DATABASE[HC_COLS], on=['Date', 'IEX_ID'], how='left')
    .sort_values(['Date', 'IEX_ID', 'Datetime_Start_Action'], ascending=True)
    .reset_index(drop=True)
    .drop_duplicates()
)

In [58]:
# ── LABEL BREAKS & FILL ZEROS ─────────────────────────────────────────────────

sorted_df = (
    sorted_df
    .groupby(['Date', 'IEX_ID'], group_keys=False)
    .apply(label_breaks)
    .reset_index(drop=True)
)

FILL_ZERO = [
    'Time_Of_Day', 'Open Time', 'Extra Time', 'Target',
    'Break Time', 'Lunch Time', 'Training', 'NCNS', 'AL',
    'Night_Shift', 'OT PreShift', 'OT PostShift',
]
sorted_df[FILL_ZERO] = sorted_df[FILL_ZERO].fillna(0)

# Roster Scheduled / Presented (vectorised)
productive_total = sorted_df['Open Time'] + sorted_df['Extra Time'] + sorted_df['Training']

sorted_df['Roster Scheduled'] = np.where(
    (sorted_df['Time_Of_Day'] < 6) & (sorted_df['Scheduled'] == 1), 0.5,
    sorted_df['Scheduled']
)
sorted_df['Roster Presented'] = np.select(
    [productive_total == 0, (productive_total > 0) & (productive_total < 6)],
    [0, 0.5],
    default=1,
)

# Unplanned / Planned
sorted_df['Unplanned'] = np.select(
    [sorted_df['NCNS'] == 0, (sorted_df['NCNS'] > 0) & (sorted_df['NCNS'] < 6)],
    [0, 0.5], default=1
)
sorted_df['Planned'] = np.select(
    [sorted_df['AL'] == 0, (sorted_df['AL'] > 0) & (sorted_df['AL'] < 6)],
    [0, 0.5], default=1
)

# OT Type (vectorised)
sorted_df['OT Type'] = determine_ot_type(
    sorted_df['Shift Tracking'],
    sorted_df['OT PreShift'],
    sorted_df['OT PostShift'],
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13744\500800625.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(label_breaks)


In [59]:
# ── DEBUG CHECK ───────────────────────────────────────────────────────────────

sorted_df[
    (sorted_df['IEX_ID'] == '3026480') &
    (sorted_df['Week_Monday'] == pd.Timestamp('2026-04-27'))
]

,sheet_name,Generate Date,Year,Month,Week Begin,Week_Monday,Date,Start Date,End Date,Agent Name,...,LOB_3,Supervisor Name,Wave,Detail Status,Status,Roster Scheduled,Roster Presented,Unplanned,Planned,OT Type
49772,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-27,2026-04-27,2026-04-27,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,1.0,0.0,0.0,No OT
49773,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-27,2026-04-27,2026-04-27,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,1.0,0.0,0.0,No OT
49774,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-27,2026-04-27,2026-04-27,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,1.0,0.0,0.0,No OT
49775,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-27,2026-04-27,2026-04-27,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,1.0,0.0,0.0,No OT
49776,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-27,2026-04-27,2026-04-27,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,1.0,0.0,0.0,No OT
49777,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-27,2026-04-27,2026-04-27,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,1.0,0.0,0.0,No OT
49778,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-27,2026-04-27,2026-04-27,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,1.0,0.0,0.0,No OT
50243,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-28,2026-04-28,2026-04-28,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,0.0,1.0,0.0,No OT
50708,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-29,2026-04-29,2026-04-29,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,1.0,0.0,0.0,No OT
50709,IEX_Base_2026_04_27,2026-05-05 03:39:00,2026,Apr-26,WB2704,2026-04-27,2026-04-29,2026-04-29,2026-04-29,"NGO, TANPHAT",...,None,Patar Kirpan,39,Lodging Nesting,Active,1.0,1.0,0.0,0.0,No OT


In [60]:
# ── GENERATE 30-MIN INTERVALS ─────────────────────────────────────────────────

def generate_intervals_extend(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    Expand each activity row into 30-minute interval rows.
    Fully vectorised — no Python-level loops over rows.
    """
    df = df_input.copy()

    df['Datetime_Start_Action'] = pd.to_datetime(df['Datetime_Start_Action'], errors='coerce')
    df['Datetime_End_Action']   = pd.to_datetime(df['Datetime_End_Action'],   errors='coerce')
    df = df.dropna(subset=['Datetime_Start_Action', 'Datetime_End_Action'])

    # Fix overnight (End < Start)
    overnight = df['Datetime_End_Action'] < df['Datetime_Start_Action']
    df.loc[overnight, 'Datetime_End_Action'] += pd.Timedelta(days=1)

    df['_floor'] = df['Datetime_Start_Action'].dt.floor('30min')
    df['_ceil']  = df['Datetime_End_Action'].dt.ceil('30min')
    df['_n']     = (((df['_ceil'] - df['_floor']) / pd.Timedelta(minutes=30)).fillna(0).astype(int))
    df = df[df['_n'] > 0].copy()

    # Expand
    df_exp = df.loc[df.index.repeat(df['_n'])].copy()
    df_exp['_i'] = df_exp.groupby(level=0).cumcount()

    df_exp['Datetime_Start_Time_Full'] = df_exp['_floor'] + pd.to_timedelta(df_exp['_i'] * 30, unit='m')
    df_exp['Datetime_End_Time_Full']   = df_exp['Datetime_Start_Time_Full'] + pd.Timedelta(minutes=30)

    # Clip to actual action bounds
    df_exp['Datetime_Start_Time'] = np.maximum(df_exp['Datetime_Start_Time_Full'], df_exp['Datetime_Start_Action'])
    df_exp['Datetime_End_Time']   = np.minimum(df_exp['Datetime_End_Time_Full'],   df_exp['Datetime_End_Action'])
    df_exp = df_exp[df_exp['Datetime_Start_Time'] < df_exp['Datetime_End_Time']].copy()

    df_exp['Duration'] = (
        df_exp['Datetime_End_Time'] - df_exp['Datetime_Start_Time']
    ).dt.total_seconds() / 3600

    # Timezone columns
    df_exp['VNT_Intervals'] = df_exp['Datetime_Start_Time_Full']
    df_exp['PST_Intervals'] = (
        df_exp['VNT_Intervals']
        .dt.tz_localize('Asia/Ho_Chi_Minh')
        .dt.tz_convert('America/Los_Angeles')
        .dt.tz_localize(None)
    )

    v_s = df_exp['VNT_Intervals'].dt.strftime('%H:%M')
    v_e = (df_exp['VNT_Intervals'] + pd.Timedelta(minutes=30)).dt.strftime('%H:%M')
    p_s = df_exp['PST_Intervals'].dt.strftime('%H:%M')
    p_e = (df_exp['PST_Intervals'] + pd.Timedelta(minutes=30)).dt.strftime('%H:%M')

    df_exp['VNT_Interval_Range'] = v_s + '-' + v_e
    df_exp['PST_Interval_Range'] = p_s + '-' + p_e
    df_exp['Intervals']          = df_exp['VNT_Intervals'].dt.time

    df_exp['Work Category'] = np.where(
        df_exp['Scheduled Activity'].isin(['Open Time', 'Extra Hours']),
        'Productive', 'Unproductive'
    )

    df_exp = df_exp.rename(columns={'Date': 'Date_Converted', 'IEX_ID': 'IEX ID'})

    TARGET_COLS = [
        'Month', 'Week_Monday', 'Date_Converted', 'Employee Name', 'OracleID',
        'IEX ID', 'Email Id', 'First Shift', 'Scheduled Activity',
        'VNT_Intervals', 'PST_Intervals', 'VNT_Interval_Range', 'PST_Interval_Range',
        'Datetime_Start_Time', 'Datetime_End_Time', 'Duration', 'Work Category',
    ]
    return df_exp[[c for c in TARGET_COLS if c in df_exp.columns]].reset_index(drop=True)


INTERVALS_EXTEND = generate_intervals_extend(sorted_df)

In [61]:
# ── BUILD FINAL OUTPUT TABLES ─────────────────────────────────────────────────

SCHEDULE = (
    sorted_df
    .pivot_table(
        index=['OracleID', 'People ID', 'IEX_ID', 'Employee Name', 'Email Id',
               'Alias', 'LOB', 'LOB_2', 'LOB_3', 'Supervisor Name',
               'Wave', 'Detail Status', 'Status'],
        columns='Date',
        values='First Shift',
        aggfunc='first',
    )
    .reset_index()
)

IEX_EXTEND_COLS = [
    'Year', 'Month', 'Week Begin', 'Week_Monday', 'Date', 'Agent Name', 'IEX_ID',
    'Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift', 'Fluctuate Shift',
    'Datetime_First_Start_Shift', 'Datetime_First_End_Shift', 'First Shift',
    'Time_Of_Day', 'Open Time', 'Extra Time', 'Target',
    'Break Time', 'Lunch Time', 'Training', 'NCNS', 'AL',
    'Unplanned', 'Planned', 'Night_Shift', 'Roster Presented', 'Roster Scheduled',
    'Shift Tracking', 'OT PreShift', 'OT PostShift',
    'OracleID', 'People ID', 'Employee Name', 'Email Id', 'Alias',
    'LOB', 'LOB_2', 'LOB_3', 'Supervisor Name', 'Wave', 'Detail Status', 'Status',
    'OT Type', 'Combined OT Range',
]
IEX_EXTEND = (
    sorted_df[[c for c in IEX_EXTEND_COLS if c in sorted_df.columns]]
    .drop_duplicates()
)

In [62]:
# ── GET VALID WEEKS FROM input_iex ────────────────────────────────────────────
input_iex_files = (
    glob.glob(f'{folder_paths["input_iex"]}/*.csv') +
    glob.glob(f'{folder_paths["input_iex"]}/*.xlsx')
)

valid_weeks = {
    pd.to_datetime(Path(f).stem.replace('_', '-'), errors='coerce')
    for f in input_iex_files
}
valid_weeks.discard(pd.NaT)
print(f"Valid weeks: {sorted(valid_weeks)}")

# ── EXPORT TO CSV ─────────────────────────────────────────────────────────────

for week_monday, group in IEX_EXTEND.groupby('Week_Monday'):
    if pd.Timestamp(week_monday) not in valid_weeks:
        continue
    wm_str = week_monday.strftime('%Y_%m_%d')
    for out_dir in (folder_paths['output_iex'], folder_paths['output_iex_all']):
        group.to_csv(os.path.join(out_dir, f'{wm_str}.csv'), index=False)

for week_monday, group in INTERVALS_EXTEND.groupby('Week_Monday'):
    if pd.Timestamp(week_monday) not in valid_weeks:
        continue
    wm_str = week_monday.strftime('%Y-%m-%d')
    group.to_csv(os.path.join(folder_paths['output_iex_intervals'], f'{wm_str}.csv'), index=False)

Valid weeks: [Timestamp('2026-01-05 00:00:00'), Timestamp('2026-01-12 00:00:00'), Timestamp('2026-01-19 00:00:00'), Timestamp('2026-01-26 00:00:00'), Timestamp('2026-02-02 00:00:00'), Timestamp('2026-02-09 00:00:00'), Timestamp('2026-02-16 00:00:00'), Timestamp('2026-02-23 00:00:00'), Timestamp('2026-03-02 00:00:00'), Timestamp('2026-03-09 00:00:00'), Timestamp('2026-03-16 00:00:00'), Timestamp('2026-03-23 00:00:00'), Timestamp('2026-03-30 00:00:00'), Timestamp('2026-04-06 00:00:00'), Timestamp('2026-04-13 00:00:00'), Timestamp('2026-04-20 00:00:00'), Timestamp('2026-04-27 00:00:00'), Timestamp('2026-05-04 00:00:00'), Timestamp('2026-05-11 00:00:00'), Timestamp('2026-05-18 00:00:00')]
